# Final Assignment4 - Green Patent Detection: Advanced Agentic Workflow with QLoRA (Option: Mistral-7B and LangGraph)
> Suchanya Baiyam Business Data Science AAU
 - Build a state-of-the-art data labeling pipeline
 - Fine-tune an LLM using QLoRA to understand patent language
 - Integrate that fine-tuned model into a Multi-Agent System (MAS) to debate and label complex claims


Part A & B: Set up
- Dataset: Load patents_50k_green.parquet (the balanced 50k sample).
- Uncertainty Sampling: Re-use the uncertainty scores u from my baseline model.
- Export: Select the same top 100 high-risk claims I used previously.

STEP01: Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
df = pd.read_parquet("patent_50k_green.parquet")
df.shape

(50000, 4)

In [ ]:
high_risk_100 = pd.read_csv("hitl_green_100.csv")
high_risk_100.shape

(100, 9)

Part C STEP01: Domain Adaptation (QLoRa)
- Fine-tune a Generative LLM (Mistral-7B) using QLoRa on train_silver set
-  The goal is to adapt the model to the dense linguistic style of patent claims and the logic of Y02 classifications

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 48.4 MB/s eta 0:00:00


STEP01: Load Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

STEP02: Load Model (4bit QLora)

In [ ]:
#assisted by chatgpt
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

STEP03: LoRa Config

In [ ]:
from peft import LoraConfig, get_peft_model
#assisted by chatgpt but r, alpha, target_modules are project's design choice
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 4,718,592 || all params: 7,252,742,144 || trainable%: 0.0651


STEP04: Generation Settings (max_leghth = 400, tempurature = 0.3)

In [ ]:
#code assisted by chatGPT
generation_config = {"max_new_tokens":400, "tempurature": 0.3, "top_p": 0.9, "do_sample": True}

TEST before training

In [ ]:
#code by chatGPT
prompt = "Explain whether a patent claim about recyclable materials could be considered environmentally friendly."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.3
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Explain whether a patent claim about recyclable materials could be considered environmentally friendly.

A patent claim about recyclable materials could be considered environmentally friendly because recycling is a process that reduces waste and conserves resources. By using recyclable materials, the invention is contributing to the reduction of waste in landfills, the conservation of natural resources, and the decrease in greenhouse gas emissions associated with the extraction, production, and disposal of new materials.

However, it's important to note that not all recyclable materials are necessarily environmentally friendly. For example, recycling certain materials, such as plastic bags, can still have negative environmental impacts due to the energy and resources required for the recycling process, as well as the potential for contamination of other recyclable materials.

Additionally, the environmental benefits of recycling depend on various factors, such as the efficiency of the 

STEP05: Save Adapter

In [ ]:
model.save_pretrained("lora_green_patent")
tokenizer.save_pretrained("lora_green_patent")

('lora_green_patent/tokenizer_config.json',
 'lora_green_patent/chat_template.jinja',
 'lora_green_patent/tokenizer.json')

save to continue on next colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model.save_pretrained("/content/drive/MyDrive/green_patent_lora")